# Training Strategies for Neural Networks

In the Artificial Neural Networks repository you trained networks with the settings you were given. Those settings were choices: the activation function, the optimiser, the batch size and how the weights start. In this notebook you change one setting at a time on the same small dataset and compare the learning curves, so you can see what each choice does. Notebook 02 then deals with overfitting and regularisation.

## Learning Objectives

By the end of this notebook, you should be able to:

- Choose the activation function and the loss of the output layer for regression, binary and multiclass problems.
- Compare hidden activations, optimisers and batch sizes by their validation learning curves.
- Compare weight initialisers, and add batch normalisation to a deep Keras network so that it trains.
- Choose the settings of a new network from a short list of recommended defaults.

## Setup

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)  # fix the random seed so every run gives the same data

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. The Settings That Shape Training

The weights of a network are learnt. Everything else is a **hyperparameter** that you choose before training:

| Setting | What it controls | Default in this notebook |
|---|---|---|
| Layers and neurons | How complex a function the network can build | 2 hidden layers of 32 neurons |
| Activation function | How each neuron transforms its weighted sum | ReLU in hidden layers |
| Loss | What training minimises | depends on the problem (section 2) |
| Optimiser and learning rate | How the weights are updated from the gradients | Adam, learning rate 0.01 (the Keras default is 0.001) |
| Batch size | How many examples are used for one update | 32 |
| Weight initialisation | Where training starts | Glorot uniform (the Keras default) |
| Batch normalisation | Whether the inputs of each layer are rescaled during training | off |
| Regularisation | Whether the network is stopped from memorising the training data | off (notebook 02) |

The data forms a bowl-shaped surface: two input features $x_1$ and $x_2$ between -3 and 3, and the target $y = x_1^2 + x_2^2$ plus noise. It is small enough to train many networks in a few minutes. The learning rate of 0.01, ten times the Keras default, lets these small networks reach their final level within 100 epochs.

Every comparison in this notebook is a choice between settings, so it is made on a **validation set**. Split the 500 examples with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) into 400 training and 100 validation examples.

The cell also prints a reference value: the validation RMSE of a model that ignores the inputs and predicts the mean of the training targets for every example. A network that has not learnt anything useful ends at this value, so it is the level every network has to beat clearly.

In [ ]:
bowl = pd.DataFrame({"x1": rng.uniform(-3, 3, size=500), "x2": rng.uniform(-3, 3, size=500)})
bowl["y"] = bowl["x1"] ** 2 + bowl["x2"] ** 2 + rng.normal(0, 0.5, size=500)

# TODO: split bowl[["x1", "x2"]] and bowl["y"] into a training set and a validation set of 100 examples,
# with random_state=42
X_train, X_valid, y_train, y_valid = ...
print(f"Training set: {X_train.shape}, validation set: {X_valid.shape}")

mean_rmse = np.sqrt(((y_valid - y_train.mean()) ** 2).mean())  # always predict the mean of the training targets
print(f"Validation RMSE when predicting the mean: {mean_rmse:.3f}")

Every experiment builds a model, trains it and compares it with the others, so the Keras steps you already know (configure, compile, fit, evaluate) are wrapped in functions. `build_model` creates a new model with new random weights every time it is called; each argument is one of the settings in the table. With batch normalisation, the activation becomes a separate [`Activation`](https://keras.io/api/layers/core_layers/activation/) layer (section 4). `train_model` compiles and trains it and returns its learning curves; the [`RootMeanSquaredError`](https://keras.io/api/metrics/regression_metrics/#rootmeansquarederror-class) metric reports the RMSE after every epoch. `compare` plots the validation curves of several runs and lists their final values.

In [ ]:
def build_model(units=(32, 32), activation="relu", initializer="glorot_uniform", batch_norm=False):
    """Return a new regression network with the given hidden layers and settings."""
    keras.utils.clear_session()  # start from a clean Keras state
    keras.utils.set_random_seed(42)  # the same starting weights for the same settings
    layers = [keras.Input(shape=(2,))]
    for n_neurons in units:
        if batch_norm:  # weighted sum, then batch normalisation, then the activation
            layers.append(keras.layers.Dense(units=n_neurons, kernel_initializer=initializer))
            layers.append(keras.layers.BatchNormalization())
            layers.append(keras.layers.Activation(activation))
        else:
            layers.append(
                keras.layers.Dense(units=n_neurons, activation=activation, kernel_initializer=initializer)
            )
    layers.append(keras.layers.Dense(units=1))  # regression output: one neuron, no activation
    return keras.Sequential(layers)

In [ ]:
def train_model(model, optimizer=None, epochs=100, batch_size=32):
    """Compile and train a model; return its training and validation curves as a DataFrame."""
    model.compile(
        loss="mse",
        optimizer=optimizer or keras.optimizers.Adam(learning_rate=0.01),
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    history = model.fit(
        X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_valid, y_valid), verbose=0
    )
    return pd.DataFrame(history.history)


def compare(runs, title):
    """Plot the validation curve of every run and return a table of the final RMSEs."""
    for name, logs in runs.items():
        plt.plot(logs["val_rmse"], label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation RMSE")
    plt.title(title)
    plt.legend()
    plt.show()
    return pd.DataFrame(
        {
            "final training RMSE": {name: logs["rmse"].iloc[-1] for name, logs in runs.items()},
            "final validation RMSE": {name: logs["val_rmse"].iloc[-1] for name, logs in runs.items()},
        }
    ).round(3)

## 2. Activation Functions and Losses

The output layer depends on the problem, and so does the loss that goes with it. The hidden layers are where you have a choice:

| Layer | Activation | Loss | Example |
|---|---|---|---|
| Hidden layers | ReLU (default), ELU, tanh | | most networks you build |
| Output, regression | none (linear) | `"mse"` | the bowl, house prices |
| Output, binary classification | sigmoid | `"binary_crossentropy"` | spam or not spam |
| Output, multiclass classification | softmax, one neuron per class | `"sparse_categorical_crossentropy"` | one of seven tree species from terrain data |

For multiclass problems, the **sparse** version of the loss takes the labels as integers (0, 1, 2, ...); if the labels are one-hot encoded vectors, use `"categorical_crossentropy"` instead. Both compute the same categorical cross-entropy.

The figure shows the four functions you meet most often in hidden layers (solid) and their slopes (dashed). The slope matters because backpropagation multiplies one slope per layer on the way back to the first layer.

<img src="assets/activation_functions.png" width="900">

- **Sigmoid** maps any number into the range 0 to 1, which makes it a good output for a probability, but a poor choice for hidden layers. Its slope is at most 0.25, so after a few layers the gradients of the early layers become very small: the **vanishing gradient** problem.
- **Tanh** maps any number into the range -1 to 1 and has a slope of up to 1, but its slope also approaches 0 for large inputs.
- **ReLU** returns the input when it is positive and 0 otherwise. Its slope is exactly 1 wherever it is active, so gradients pass through unchanged.
- **ELU** follows ReLU for positive inputs and approaches -1 smoothly for negative ones, so a neuron with a negative input still gets a small gradient.

[Softmax](https://keras.io/api/layers/activations/#softmax-function) is not drawn because it works on a whole layer at once: it turns the outputs of the last layer into probabilities that sum to 1, one per class. The cell below trains the same network with four hidden activations. Before you run it, predict which activation learns slowest, and why.

In [ ]:
runs = {name: train_model(build_model(activation=name)) for name in ["relu", "elu", "tanh", "sigmoid"]}
compare(runs, "Hidden activation")

After 100 epochs the four activations end close together, with final validation RMSEs between 0.50 and 0.54, but they differ in how quickly they reach it. ReLU and ELU fall fastest and are near their final level after about 10 epochs. Tanh stays on a plateau for about 8 epochs first. Sigmoid shows the vanishing gradient: its validation RMSE stays at about 3.9 for the first 25 epochs, the RMSE of predicting the mean, because almost no gradient reaches the first layer; only then does it drop. With two hidden layers sigmoid reaches a similar level by the end; section 4 shows the effect with six.

## 3. Optimisers and Batch Size

### Gradient Descent in Batches

Gradient descent can compute each update from a different number of examples:

| Variant | Examples per update | Updates per epoch here (400 training examples) | Behaviour |
|---|---|---|---|
| Batch gradient descent | all of them | 1 | smooth but slow, and needs the whole dataset in memory |
| Stochastic gradient descent | one | 400 | fast but noisy updates |
| Mini-batch gradient descent | a few, set by `batch_size` | 13 with `batch_size=32` | the usual compromise, and what Keras does |

The gradient descent you wrote by hand in the Artificial Neural Networks repository used the whole dataset for every update. Keras uses mini-batches.

### Optimisers

The optimiser turns the gradients into an update:

- **SGD** takes a step of the learning rate times the gradient. Despite its name, the Keras [`SGD`](https://keras.io/api/optimizers/sgd/) optimiser is only the update rule: how many examples each update uses is set separately by `batch_size`, so with `batch_size=32` it performs mini-batch gradient descent.
- **Momentum** adds part of the previous step to the current one, so the updates keep moving in a consistent direction and accelerate along it.
- **RMSprop** divides each step by a running average of the size of recent gradients, so every weight gets its own step size.
- **Adam** combines momentum and RMSprop. It is the usual default.

The cell below trains the same network with four [optimisers](https://keras.io/api/optimizers/), all with a learning rate of 0.01. Before you run it, predict which optimiser falls slowest at the start.

In [ ]:
optimisers = {
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "SGD with momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": keras.optimizers.RMSprop(learning_rate=0.01),
    "Adam": keras.optimizers.Adam(learning_rate=0.01),
}
runs = {name: train_model(build_model(), optimizer=optimiser) for name, optimiser in optimisers.items()}
compare(runs, "Optimiser")

The optimisers differ both in speed and in where they end. SGD with momentum falls fastest at the start and is close to its final value after about 10 epochs, while plain SGD decreases slowly and ends highest (validation RMSE 0.648). RMSprop oscillates strongly between epochs 15 and 30, because its step sizes are large for this learning rate. Adam is smooth throughout and ends with the lowest validation RMSE (0.540), which is why it is the usual default.

### Batch Size

The batch size sets how many examples each update uses, and therefore how many updates one epoch makes. The cell below trains the same network for 100 epochs with three batch sizes. Before you run it, predict which batch size gives the noisiest curve.

In [ ]:
runs = {f"batch size {size}": train_model(build_model(), batch_size=size) for size in [8, 32, 128]}
compare(runs, "Batch size")

A batch size of 8 makes 50 updates per epoch, so its validation RMSE falls fastest in the first few epochs, but its noisy updates keep it at about 0.70 for the rest of training. A batch size of 128 makes only 4 updates per epoch and needs about 40 epochs to reach the level of the others. Batch sizes of 32 and 128 end at about the same validation RMSE (0.540 and 0.533); 32 reaches it much sooner.

## 4. Initialisation and Batch Normalisation

### Weight Initialisation

The weights must start random: neurons with identical weights compute the same output, receive identical gradients and stay identical. The scale of the random values matters too. If they are too large or too small, the outputs and the gradients grow or shrink from layer to layer. Two [initialisers](https://keras.io/api/layers/initializers/) pick the scale from the number of inputs and outputs of each layer:

- **Glorot** (also called Xavier) initialisation, the Keras default, suits tanh and sigmoid.
- **He** initialisation uses slightly larger values and is designed for ReLU.

The cell below compares them with all weights starting at zero. Before you run it, predict what happens when every weight starts at zero.

In [ ]:
runs = {name: train_model(build_model(initializer=name)) for name in ["zeros", "glorot_uniform", "he_normal"]}
compare(runs, "Weight initialisation")

With all weights at zero the network does not learn: after 100 epochs its validation RMSE is still 3.946, the value of predicting the mean. With zero weights, every hidden neuron outputs 0, and ReLU passes no gradient back for an input of 0, so none of the hidden weights ever changes. Only the bias of the output neuron learns, and it converges to the mean of the targets. Glorot and He both work and end with almost the same validation RMSE (0.540 and 0.552). On a network with two hidden layers the difference between them is small; He initialisation is mainly beneficial in deeper ReLU networks.

### Batch Normalisation

A [`BatchNormalization`](https://keras.io/api/layers/normalization_layers/batch_normalization/) layer rescales the weighted sums of each mini-batch to a mean of about 0 and a standard deviation of about 1, then applies a learnt scale and shift. This keeps the inputs of every layer in a range where the activation still has a useful slope, which helps deep networks train. It is placed between the weighted sum and the activation.

The cell below trains a deep network of six sigmoid layers, the setting where vanishing gradients have the largest effect, without and with batch normalisation. Both use plain SGD with a learning rate of 0.1.

In [ ]:
deep = {"units": (32, 32, 32, 32, 32, 32), "activation": "sigmoid"}
runs = {
    "without batch normalisation": train_model(build_model(**deep), optimizer=keras.optimizers.SGD(learning_rate=0.1)),
    "with batch normalisation": train_model(
        build_model(**deep, batch_norm=True), optimizer=keras.optimizers.SGD(learning_rate=0.1)
    ),
}
rmse_deep = runs["without batch normalisation"]["val_rmse"].iloc[-1]  # used again in the Try It
compare(runs, "Deep sigmoid network")

Without batch normalisation the deep sigmoid network does not learn: its validation RMSE stays at about 3.9 for all 100 epochs, the RMSE of predicting the mean: the vanishing gradient again. With batch normalisation the RMSE first rises for about 12 epochs, most likely because the first updates with a learning rate of 0.1 are too large while the rescaling of each layer is still changing, then falls steeply between epochs 15 and 40 and ends at 1.599. It is still far from the shallow networks above, but batch normalisation has made a network trainable that otherwise does not learn at all.

## 5. Recommended Starting Settings

When you build a new network, these defaults are a good starting point:

- Check the shape of the input layer: one input per feature.
- Choose the output layer and the loss from the table in section 2.
- Use ReLU in the hidden layers, with He initialisation if the network is deep.
- Use Adam with its default learning rate of 0.001, and a batch size of 32. A larger learning rate, as in this notebook, trains faster on small problems but can make the curves jump.
- Watch the training and validation curves. If both stay high, make the network larger or train longer; if they separate, regularise the network (notebook 02).
- Change one setting at a time, so you know which change made the difference.

## Try It: Make the Deep Network Learn

The deep network of section 4, six layers of 32 sigmoid neurons trained with SGD, ended with a validation RMSE of 3.912 without batch normalisation, close to the RMSE of predicting the mean. Make the same six layers learn, in two steps:

1. Keep the sigmoid activation and batch normalisation, and replace SGD with Adam (learning rate 0.01).
2. Choose all settings yourself, using the defaults of section 5.

Aim for a validation RMSE far below the 3.9 of predicting the mean in step 1, and below 1.0 in step 2; the shallow networks of sections 2 to 4 reached about 0.54.

In [ ]:
# Step 1: sigmoid with batch normalisation, trained with Adam
model_bn = build_model(units=deep["units"], activation="sigmoid", batch_norm=True)
# TODO: train model_bn with train_model and an Adam optimiser with a learning rate of 0.01
logs_bn = ...
print(f"Step 1: validation RMSE {logs_bn['val_rmse'].iloc[-1]:.3f}")

# Step 2: your own settings
# TODO: choose the hidden activation: "sigmoid", "tanh" or "relu"
my_activation = ...
# TODO: choose the weight initialiser: "glorot_uniform" or "he_normal"
my_initializer = ...
# TODO: switch batch normalisation on or off (True or False)
my_batch_norm = ...
# TODO: choose an optimiser and its learning rate
my_optimizer = ...

model_try = build_model(
    units=deep["units"], activation=my_activation, initializer=my_initializer, batch_norm=my_batch_norm
)
logs_try = train_model(model_try, optimizer=my_optimizer)
print(f"Step 2: validation RMSE {logs_try['val_rmse'].iloc[-1]:.3f} (deep sigmoid network with SGD: {rmse_deep:.3f})")

## Summary

- The output layer and the loss follow from the problem: no activation and MSE for regression, sigmoid and binary cross-entropy for two classes, softmax and sparse categorical cross-entropy for more (with integer labels).
- ReLU is the default for hidden layers; sigmoid hidden layers suffer from vanishing gradients.
- Keras trains with mini-batches. The optimiser turns the gradients into updates: momentum and RMSprop speed up plain SGD, and Adam combines both.
- The weights must start random and at a suitable scale: Glorot by default, He for deep ReLU networks. Batch normalisation helps deep networks train.
- Settings are compared on a validation set, one change at a time, by their learning curves.

Next, continue with [02 - Overfitting and Regularisation](02_overfitting_and_regularisation.ipynb).

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 11: Training Deep Neural Networks.
- Keras API: [activations](https://keras.io/api/layers/activations/), [optimisers](https://keras.io/api/optimizers/), [layer weight initialisers](https://keras.io/api/layers/initializers/) and [`BatchNormalization`](https://keras.io/api/layers/normalization_layers/batch_normalization/).
- deeplearning.ai. [Initializing neural networks](https://www.deeplearning.ai/ai-notes/initialization/index.html): an interactive explanation of why the scale of the starting weights matters.
- Ruder, S. (2016). [An overview of gradient descent optimization algorithms](https://www.ruder.io/optimizing-gradient-descent/).
- [TensorFlow Playground](https://playground.tensorflow.org/): try the activation and the learning rate in the browser.
- The structure follows the training strategies notebook of the Spiced Academy deep learning course.